# P11.1 – P11.2b — Pruebas técnicas, lotes de lenguaje real y pre-piloto
## (a) Paso del protocolo y objetivo
- **P11.1** (pruebas técnicas internas): `rasa data validate`, *smoke test* de las 54 intenciones y validación cruzada, sin personas externas.
- **Lotes 1 y 2** (compuertas **G1, G2**): ingesta de las frases reales, revisión de etiquetas y de datos personales.
- **P11.2b** (pre-piloto, compuerta **G6**): sesiones con el **asistente local** y el modelo congelado; criterio: 5–15 sesiones completas y **alfa de Cronbach ≥ 0,70** (encuesta de 9 ítems; ítems 1–8 para el alfa).
Sustentan el **OE2** (el chatbot funciona con personas reales) y preparan el **OE4** (satisfacción, con la encuesta de 9 ítems).
**Origen:** P11.1 sobre el corpus **Sintético**; lotes y pre-piloto **Reales** (solo agregados); pruebas automáticas con datos **Simulados** (falsos).

## (b) Código del repositorio que lo implementa
`smoke_test.py`, `plot_p11_1.py`, `plot_v3_comparacion.py` (P11.1); `ingest_real_lote.py`, `trasladar_revision.py`, `conciliar_seguimiento.py`, `hoja_revision_lote2.py`, `informe_ingesta_lote2.py`, `catalogo_formularios.py`, `preparar_libro_v13.py`, `ampliar_libro.py`, `combinar_libros.py`, `libro_xml.py` (lotes); `asistente_local.py`, `preparar_registro_prepiloto.py`, `analizar_piloto.py` (**documentado, NO ejecutado** sobre datos reales), `estado_compuertas.py` (tablero). Entradas y salidas: ver tabla.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("11"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,ampliar_libro.py,T03 (lote real),Agrega a un libro de transcripción (vacío o ya lleno) las filas de un complemento: participantes nuevos con su formulario y una fila de Respuestas por cada situación.,Libro de transcripción + complemento,Libro ampliado (participantes y situaciones nuevas),"python scripts/ampliar_libro.py --base <libro> --salida <libro nuevo> --codigos P29,P30,P31 --forma G --situaciones S46,S47",No,Documentado
1,analizar_piloto.py,"P12–P14 (piloto), usa el registro de sesiones","Análisis del piloto exploratorio (protocolo V1.3: una sesión asistida, n = 60 sesiones elegibles).",Registro de sesiones (.xlsx) elegibles,"Análisis del piloto sobre sesiones elegibles: tiempos, alfa de Cronbach, prueba final del modelo (UNA vez)",python scripts/analizar_piloto.py --registro docs/piloto/privado/Registro_lleno.xlsx --modelo-congelado logs/v3_real/modelo_congelado.json \,No,Documentado — NO ejecutado sobre datos reales (gastaría la prueba final única)
2,asistente_local.py,"P11.2b (pre-piloto), T08","Asistente de consola para el PRE-PILOTO (protocolo V1.8, 2.12 y T08: «asistente local en una laptop» con el modelo congelado).",Modelo congelado + domain_v3.yml,Consola que responde con utter_<intención> o «no entendí»; log por sesión en carpeta privada,python scripts/asistente_local.py PP01,Sí (import),Documentado (requiere Rasa; NO se ejecuta aquí)
3,catalogo_formularios.py,T03 (lote real),"Formularios de cada situación del lote 1, con el complemento del lote 1b (formulario F).",—,Catálogo de formularios A–F del lote 1,python scripts/catalogo_formularios.py --help,No,Documentado
4,combinar_libros.py,T03 (lote real),"Combina en un libro de transcripción (BASE) los datos de otro libro (FUENTE): solo las celdas de ENTRADA de Participantes y Respuestas, editando el XML de la base (no se guarda con openpyxl: se conservan valida",Libro BASE y libro FUENTE,BASE con celdas de entrada de la FUENTE (edición del XML),python scripts/combinar_libros.py --base <libro con P26–P28> --fuente <libro con P01–P25> --salida docs/lote_real_1/privado/<nombre>.xlsx,No,Documentado
5,conciliar_seguimiento.py,T03.1 (lote 1),Conciliación del seguimiento del lote 1 (xlsx) con las respuestas transcritas (CSV).,Seguimiento del lote 1 + respuestas,Conciliación y alertas de perfil,python scripts/conciliar_seguimiento.py --seguimiento docs/lote_real_1/privado/Seguimiento_Lote1_Participantes_real.xlsx,No,Documentado
6,entrenar_modelo_demo.py,Demostración (P08),"Entrena el modelo de DEMOSTRACIÓN del asistente: Rasa/DIET con la configuración del modelo congelado, pero SOLO con el corpus SINTÉTICO (data/nlu_train.yml, 546 frases).",data/nlu_train.yml (sintético) + configs/rasa_config_lote2.yml (copia),models/demo_vivo/DEMO-VIVO.tar.gz,python scripts/entrenar_modelo_demo.py # configuración oficial: 100 épocas (unos 5 min),Sí (python -m rasa),Documentado (requiere Rasa; solo datos sintéticos)
7,estado_compuertas.py,2.14 (G1–G7),"Tablero de las compuertas de avance (protocolo V1.4, sección 2.14, G1 a G7).",Archivos de evidencia de logs/ y docs/,"logs/avance/estado_compuertas.{json,md}",python scripts/estado_compuertas.py,No,Resultado guardado (etapa 12)
8,hoja_revision_lote2.py,Lote 2 (revisión),Lote 2 — hoja de revisión de etiquetas para que el tesista la llene a mano (Excel nuevo en docs/lote_real_2/privado/; trae frases reales: no se versiona ni se imprime).,Frases del lote 2 ingestadas,Excel de revisión de etiquetas (privado),python scripts/hoja_revision_lote2.py,No,Documentado (datos reales)
9,informe_ingesta_lote2.py,Lote 2,Lote 2 — informe de la ingesta SOLO con cifras e identificadores (nunca imprime frases ni observaciones).,Ingesta del lote 2,logs/avance/ingesta_lote2_cifras.md (solo cifras e ids),python scripts/informe_ingesta_lote2.py --help,No,Documentado


## (c) Celdas de código
### 1. P11.1 — Smoke test de las 54 intenciones y validación cruzada (Sintético, resultados guardados)
Requieren Rasa, por eso se muestran los resultados guardados de la ejecución del 2 de octubre de 2026 (corpus v2).

In [3]:
rotulo(ORIGEN_SINTETICO, "salida guardada de evidencias/p11_1_pruebas/salidas (no se vuelve a ejecutar: requiere Rasa)")
sm = (BASE / "evidencias/p11_1_pruebas/salidas/02_smoke_test.txt").read_text(encoding="utf-8", errors="replace")
cv = (BASE / "evidencias/p11_1_pruebas/salidas/03_rasa_test_nlu_crossval.txt").read_text(encoding="utf-8", errors="replace")
m_sm = re.search(r"(\d+)/54", sm)
print("Smoke test — intenciones correctas:", m_sm.group(0) if m_sm else "(no encontrado)")
for l in cv.splitlines():
    if re.search(r"F1|f1", l) and re.search(r"0\.7[0-9]+", l):
        print("Validación cruzada nativa:", l.strip()[:200])
        break
comparar("P11.1", "intenciones correctas en el smoke test", m_sm.group(0) if m_sm else "", "46/54", "evidencias/p11_1_pruebas/README.md", tipo="txt")
print("Lectura: la CV nativa (0,778) NO es comparable con el F1 de P11 porque los folds de Rasa no respetan base_phrase_id (fuga); el smoke test no es una métrica del protocolo (54 consultas, una por intención).")

▌ORIGEN: Sintético — salida guardada de evidencias/p11_1_pruebas/salidas (no se vuelve a ejecutar: requiere Rasa)
Smoke test — intenciones correctas: 46/54
Validación cruzada nativa: 2026-10-02 19:48:20 INFO     rasa.nlu.test  - test F1-score: 0.778 (0.028)
Lectura: la CV nativa (0,778) NO es comparable con el F1 de P11 porque los folds de Rasa no respetan base_phrase_id (fuga); el smoke test no es una métrica del protocolo (54 consultas, una por intención).


### 2. Lotes 1 y 2 — ingesta y revisión (Real, solo cifras)

In [4]:
tb = leer_json("logs/avance/estado_compuertas.json")
rotulo(ORIGEN_REAL, "compuertas G1 y G2 del tablero (ingesta del lote 1)")
display(pd.DataFrame([{"compuerta": c["id"], "estado": c["estado"], "nota": c["nota"]} for c in tb["compuertas"] if c["id"] in ("G1", "G2")]))
ing2 = (BASE / "logs/avance/ingesta_lote2_cifras.md").read_text(encoding="utf-8").splitlines()
rotulo(ORIGEN_REAL, "ingesta del lote 2 (logs/avance/ingesta_lote2_cifras.md; cifras e identificadores, sin frases)")
for l in ing2[2:9]:
    if l.strip():
        print(l[:330])
part2 = (BASE / "logs/avance/particion_lote2_cifras.md").read_text(encoding="utf-8")
print("\nRevisión y partición del lote 2:")
for l in part2.splitlines()[2:6]:
    print(l[:420])
comparar("P11.1", "participantes transcritos del lote 2", int(re.search(r"Participantes transcritos: \*\*(\d+)\*\*", "\n".join(ing2)).group(1)), 25, "ingesta_lote2_cifras.md", tol=0)

▌ORIGEN: Real — compuertas G1 y G2 del tablero (ingesta del lote 1)


,compuerta,estado,nota
0,G1,Cumplida,Transcritos 32/15 (≥ 15); intenciones con menos de 3 frases: 0.
1,G2,Cumplida,Ingesta sin errores bloqueantes y revisión de datos personales marcada.


### 3. P11.2b — Pre-piloto: sesiones, elegibilidad y alfa de Cronbach (Real, solo agregados)

### Pre-piloto (P11.2b, G6)
**Qué se hace:** a partir del registro del pre-piloto se cuentan las sesiones registradas, elegibles y no elegibles (con los motivos **agregados**), las incidencias técnicas y se calcula el **alfa de Cronbach** de los ítems 1–8 de la encuesta **paso a paso**; el resultado se verifica con una **segunda implementación independiente** (fórmula por matriz de covarianzas y, si está instalado, `pingouin`).
**Por qué:** G6 exige 5 a 15 sesiones completas y alfa ≥ 0,70; si no se cumple, el protocolo manda corregir el instrumento y repetir **una sola vez** con otro grupo.
**Paso del protocolo:** P11.2b y compuerta **G6**.
**Privacidad:** se trabaja sobre una copia **anonimizada y sin textos** (`registro_prepiloto_anonimizado.csv`: códigos `S01…`, sin nombres, edades, fechas ni frases). Solo se imprimen estadísticos agregados, **nunca filas por persona**. No se ejecuta `analizar_piloto.py`.

In [5]:
def alfa_cronbach(items):
    """α = k/(k−1) · (1 − Σ var(ítem_i) / var(suma)); varianza muestral (ddof = 1)"""
    items = np.asarray(items, dtype=float)
    k = items.shape[1]
    varianzas = items.var(axis=0, ddof=1)
    var_total = items.sum(axis=1).var(ddof=1)
    return k / (k - 1) * (1 - varianzas.sum() / var_total), varianzas, var_total


def alfa_por_covarianzas(items):
    """Segunda implementación (independiente): α = k/(k−1) · (1 − traza(C) / suma(C)), con C la matriz de covarianzas"""
    c = np.cov(np.asarray(items, dtype=float), rowvar=False)
    k = c.shape[0]
    return k / (k - 1) * (1 - np.trace(c) / c.sum())


pp = ag["prepiloto"]
if HAY_PRIVADO:
    reg = pd.read_csv(PRIV / "registro_prepiloto_anonimizado.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    registradas = len(reg)
    eleg = reg[reg["elegible"] == "Sí"]
    no_eleg = reg[reg["elegible"] == "No"]
    cols_items = [f"item{i}" for i in range(1, 9)]
    X = eleg[cols_items].astype(float).values
    a, var_i, var_t = alfa_cronbach(X)
    a2 = alfa_por_covarianzas(X)
    try:
        import pingouin
        a3 = float(pingouin.cronbach_alpha(pd.DataFrame(X))[0])
    except Exception:
        a3 = None
    rotulo(ORIGEN_REAL, "pre-piloto: solo agregados (7 sesiones registradas)")
    print(f"Sesiones registradas: {registradas} | elegibles y completas: {len(eleg)} | no elegibles: {len(no_eleg)}")
    print("Motivos de no elegibilidad (agregados):", dict(pp["motivos_no_elegibilidad"]) or "—")
    print("Incidencias técnicas declaradas:", int(reg["incidencia_tecnica"].astype(int).sum()))
    paso = pd.DataFrame({"ítem": cols_items, "media": X.mean(axis=0).round(3), "DE": X.std(axis=0, ddof=1).round(3), "varianza (ddof=1)": var_i.round(4)})
    rotulo(ORIGEN_REAL, "estadísticos descriptivos agregados de los ítems 1–8 (n = elegibles)")
    display(paso)
    print(f"Paso a paso: k = {X.shape[1]} ítems; Σ varianzas de ítems = {var_i.sum():.4f}; varianza de la suma = {var_t:.4f}")
    print(f"α = k/(k−1) · (1 − Σvar/var_total) = {X.shape[1]}/{X.shape[1] - 1} · (1 − {var_i.sum():.4f}/{var_t:.4f}) = {a:.4f}")
    print(f"Verificación independiente (covarianzas): {a2:.4f}" + (f" | pingouin: {a3:.4f}" if a3 is not None else " | pingouin no instalado (opcional)"))
    comparar("G6", "alfa de Cronbach (ítems 1–8)", a, 0.366, "Protocolo V1.8 / tablero G6 / cálculo manual del tesista", tol=5e-4)
    comparar("G6", "alfa: implementación independiente", a2, a, "misma muestra", tol=1e-9)
    comparar("G6", "sesiones registradas", registradas, 7, "tablero G6 (PP01–PP07)", tol=0)
    comparar("G6", "sesiones elegibles", len(eleg), 5, "tablero G6", tol=0)
    comparar("G6", "sesiones no elegibles", len(no_eleg), 2, "tablero G6", tol=0)
    comparar("G6", "incidencias técnicas", int(reg["incidencia_tecnica"].astype(int).sum()), 0, "registro del pre-piloto", tol=0)
    n_el, alfa_calc = len(eleg), a
else:
    requiere_privado("registro anonimizado del pre-piloto")
    rotulo(ORIGEN_REAL, "agregados GUARDADOS al construir el paquete")
    var_i, var_t, k = np.array(pp["varianzas_items"]), pp["varianza_total"], len(pp["varianzas_items"])
    a = k / (k - 1) * (1 - var_i.sum() / var_t)   # el alfa se recalcula con la fórmula desde las varianzas agregadas guardadas
    print(f"Sesiones registradas: {pp['registradas']} | elegibles: {pp['elegibles']} | no elegibles: {pp['no_elegibles']} | motivos: {pp['motivos_no_elegibilidad']} | incidencias técnicas: {pp['incidencias_tecnicas']}")
    display(pd.DataFrame({"ítem": [f"item{i}" for i in range(1, 9)], "media": pp["medias_items"], "DE": pp["de_items"], "varianza": pp["varianzas_items"]}))
    print(f"α desde las varianzas agregadas guardadas = {k}/{k - 1} · (1 − {var_i.sum():.4f}/{var_t:.4f}) = {a:.4f}")
    comparar("G6", "alfa de Cronbach desde varianzas agregadas", a, 0.366, "Protocolo V1.8 / tablero G6", tol=5e-4)
    n_el, alfa_calc = pp["elegibles"], a

aviso = f"Con n = {n_el} el alfa es poco estable (advertencia)." if n_el < 30 else ""
estado_g6 = ("En curso (menos de 5 sesiones completas)" if n_el < 5 else ("Cumplida" if alfa_calc >= 0.70 else "No cumplida"))
print(f"\nG6 (criterio: 5–15 sesiones completas y alfa ≥ 0,70): sesiones completas = {n_el}, alfa = {alfa_calc:.3f} → {estado_g6}. {aviso}")
mostrar_comparaciones(etapas=["G6"], titulo="Esta sección — recalculado frente a reportado")

▌ORIGEN: Real — pre-piloto: solo agregados (7 sesiones registradas)
Sesiones registradas: 7 | elegibles y completas: 5 | no elegibles: 2
Motivos de no elegibilidad (agregados): {'sin trámite presencial en 12 meses': 2, 'sin tiempo presencial': 2, 'sin P10': 2, 'faltan tiempos de las 3 consultas': 2, 'faltan ítems 1–8': 2, 'falta ítem 9': 2}
Incidencias técnicas declaradas: 0
▌ORIGEN: Real — estadísticos descriptivos agregados de los ítems 1–8 (n = elegibles)


,ítem,media,DE,varianza (ddof=1)
0,item1,3.8,0.447,0.2
1,item2,3.4,0.548,0.3
2,item3,4.0,0.000,0.0
3,item4,3.8,0.447,0.2
4,item5,3.4,0.548,0.3
5,item6,3.6,0.548,0.3
6,item7,4.2,0.447,0.2
7,item8,3.8,0.447,0.2


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,G6,alfa de Cronbach (ítems 1–8),0.365714,0.366000,Protocolo V1.8 / tablero G6 / cálculo manual del tesista,Sí
1,G6,alfa: implementación independiente,0.365714,0.365714,misma muestra,Sí
2,G6,sesiones registradas,7.000000,7.000000,tablero G6 (PP01–PP07),Sí
3,G6,sesiones elegibles,5.000000,5.000000,tablero G6,Sí
4,G6,sesiones no elegibles,2.000000,2.000000,tablero G6,Sí
5,G6,incidencias técnicas,0.000000,0.000000,registro del pre-piloto,Sí


### Cómo leer el resultado de esta sección
- **7 sesiones registradas, 5 elegibles y completas, 2 no elegibles, 0 incidencias técnicas.** Una sesión es «elegible y completa» solo si la persona hizo un trámite presencial en la MPSR en los últimos 12 meses, declara el tiempo presencial y P10, tiene las tres consultas con tiempo, los 8 ítems posteriores y el ítem 9 (regla de la columna «Elegible y completa» del registro).
- **α = 0,366 < 0,70 → G6 «No cumplida».** El protocolo manda corregir el instrumento y repetir una sola vez con otro grupo. Con n = 5 el alfa es **muy inestable**: un valor bajo con cinco personas no permite concluir qué ítem falla (un solo patrón de respuestas lo mueve mucho), pero tampoco se puede declarar cumplida.
- El mismo valor sale de tres vías (fórmula con varianzas, matriz de covarianzas y, si existe, `pingouin`): descarta un error de cálculo y confirma el cálculo manual del tesista.
- Solo hay estadísticos agregados; el cuaderno no muestra respuestas individuales.

### 4. El asistente local: qué hace y cómo se ejecuta (solo documentación; no se ejecuta aquí)
`asistente_local.py` es una consola para las sesiones del pre-piloto. Carga **solo el NLU congelado** (`LOTE2-FINAL v1`), verifica antes el congelamiento (`congelar_modelo.py --verificar`; si no dice «intacto», **se niega a iniciar**), aplica el umbral `t = 0,50` (y ambigüedad 0,1) y responde con `utter_<intención>` de `domain_v3.yml` o con «no entendí». Registra un CSV por sesión en `docs/piloto/privado/` (carpeta ignorada por Git) y en pantalla solo muestra la conversación (ni intención ni confianza).

**Por qué no corre en Colab:** necesita Rasa 3.6 (Python 3.10) y el modelo congelado, y escribe frases reales de personas con su código de sesión. Por eso este cuaderno solo lo documenta; su comportamiento se comprueba con la prueba automática `smoke_asistente.py` (datos falsos; resultado guardado arriba).

**Comandos** (PowerShell, desde la carpeta del repositorio; se escriben aquí como texto y no se ejecutan):

```powershell
cd "C:\Users\Luis Mario\Documents\tesis2\mpsr-chatbot\mpsr-chatbot"

# 1) SESIÓN REAL del pre-piloto (solo en la laptop del tesista): usa el siguiente código libre PPxx
.\venv\Scripts\python.exe scripts\asistente_local.py PPxx          # termina con la palabra: salir

# 2) DEMOSTRACIÓN sin material sensible (modelo entrenado SOLO con datos sintéticos; código DEMO)
.\venv\Scripts\python.exe scripts\entrenar_modelo_demo.py        # una vez (≈ 5 min); con --rapida usa 20 épocas (NO oficial)
.\venv\Scripts\python.exe scripts\asistente_local.py DEMO --demo
```

- Hay que usar `python` del entorno `venv` (3.10, con Rasa) y no `rasa.exe`: Smart App Control puede bloquearlo.
- Con `--demo` solo se acepta el código `DEMO` (los `PPxx` son sesiones reales) y el registro queda en `models/demo_vivo/logs_demo/`.
- Para ejecutar la demostración fuera de la laptop (p. ej. en un Codespace de GitHub, que sí soporta Python 3.10 y Rasa), ver `docs/Ejecutar_en_Codespaces.md`.

### 5. Pruebas automáticas y tablero de compuertas

### Pruebas automáticas
**Qué se hace:** se muestran los **resultados guardados** de las pruebas de humo del repositorio (`tests/smoke_*.py`), con qué verifica cada una y por qué importa. Las pruebas usan **datos falsos** y se ejecutaron al construir el paquete; aquí no se vuelven a correr porque dependen de la estructura completa del repositorio (y algunas de Rasa o de archivos privados).
**Por qué:** son la red de seguridad del procedimiento: comprueban que el tablero no declara avances falsos, que la prueba final no se puede gastar dos veces y que el asistente respeta el congelamiento.
**Paso del protocolo:** P15.

In [6]:
rp = leer_json("recursos/resultados_pruebas.json")
filas = []
for r in rp["pruebas"]:
    v = pr.get(r["prueba"], ["(ver docstring)", ""])
    filas.append({"prueba": r["prueba"], "resultado guardado": r["resultado"], "comprobaciones": f"{r['pass']}/{r['total']}" if r.get("total") else "—", "qué verifica": v[0], "por qué importa": v[1] if len(v) > 1 else ""})
rotulo(ORIGEN_SIMULADO, "todas las pruebas usan datos y predictores FALSOS; ejecutadas el " + rp["fecha"] + " con " + rp["entorno"])
display(pd.DataFrame(filas))
print("Suma de comprobaciones guardadas:", sum(r.get("pass", 0) for r in rp["pruebas"]), "de", sum(r.get("total", 0) for r in rp["pruebas"]))
for nombre, esperado in (("smoke_compuertas.py", (75, 75)), ("smoke_piloto.py", (51, 51)), ("smoke_asistente.py", (19, 19))):
    r = next((x for x in rp["pruebas"] if x["prueba"] == nombre), None)
    if r:
        comparar("P15", f"{nombre}", f"{r['pass']}/{r['total']}", f"{esperado[0]}/{esperado[1]}", "ejecución del 2026-10-10 (smoke_asistente suma 3 comprobaciones del modo demo)", tipo="txt")
if os.environ.get("MPSR_CORRER_PRUEBAS") == "1":
    print("(MPSR_CORRER_PRUEBAS=1 solo tiene sentido dentro del repositorio completo; no se ejecuta desde los paquetes.)")

▌ORIGEN: Simulado (demostración) — todas las pruebas usan datos y predictores FALSOS; ejecutadas el 2026-10-10 con Python 3.10.11, Rasa 3.6.21, scikit-learn 1.1.3 (venv del proyecto)


,prueba,resultado guardado,comprobaciones,qué verifica,por qué importa
0,smoke_aplicar_tupa.py,6/6 PASS,6/6,aplicar_tupa.py con un libro y un domain.yml FALSOS: el dry-run no escribe nada; --aplicar exige partición y evaluación; solo reemplaza las respuestas confirmadas y sin alerta.,Las respuestas forman parte del modelo congelado (G4 antes de G5).
1,smoke_asistente.py,19/19 PASS,19/19,Asistente de consola con un modelo de 3 épocas entrenado con corpus sintético: se niega a iniciar si el congelamiento cambió o el registro no está ignorado por Git; responde utter_<intención> o «no entendí» según el umbral; no imprime intención ni confianza; no altera el congelamiento real.,El pre-piloto usa este asistente: debe respetar el modelo congelado y la privacidad.
2,smoke_compuertas.py,75/75 PASS,75/75,"Tablero de compuertas G1–G7 con datos FALSOS: cada compuerta cambia de estado solo cuando corresponde, G6 lee solo Registro_Sesiones_Prepiloto.xlsx, lo simulado nunca cuenta como real y no se modifica corpus/real ni logs/v3_real.","Si el tablero se equivocara, se declararían avances que no existen."
3,smoke_conciliar.py,28/28 PASS,28/28,"conciliar_seguimiento.py con datos falsos: detecta respuestas ausentes, blancos con respuesta, transcritos sin respuestas y participantes no transcritos.",Evita atribuir respuestas a participantes equivocados.
4,smoke_lote2.py,51/51 PASS,51/51,"Flujo del lote 2 con datos FALSOS: puerta de congelamiento, exclusión de frases idénticas al entrenamiento solo por coincidencia exacta, evaluación única (el registro se escribe antes de predecir), IC por frases y participantes.",Asegura que la prueba independiente de G3 no se pueda contaminar ni repetir.
5,smoke_lote_real.py,88/88 PASS,88/88,"Flujo del lote 1 con datos FALSOS fuera del repositorio: ingesta, revisión, partición v3, evaluación (--smoke), umbral y dominio v3 con FallbackClassifier.",Protege el procedimiento antes de tocar datos reales.
6,smoke_piloto.py,51/51 PASS,51/51,"Análisis del piloto con datos y predictor FALSOS: lectura del registro (encabezado en fila 2 o 3), alfa de Cronbach contra numpy, prueba final única, congelamiento (--verificar detecta cambios), la plantilla real vacía no produce resultados.",Garantiza que el análisis del piloto es correcto y que no se puede gastar dos veces la prueba final.
7,smoke_transcripcion.py,51/51 PASS,51/51,"Lectura del libro de transcripción del lote 1 (ingest_real_lote.py --libro) y blancos derivados, con COPIAS del libro simulado y datos falsos.","Si la lectura del libro fallara, se perderían o confundirían datos de campo."
8,smoke_umbral_test.py,24/24 PASS,24/24,"fallback_threshold.py --fase test: aplica el umbral congelado a predicciones de test falsas ya guardadas, sin reentrenar ni cargar ningún modelo.",El umbral no puede elegirse mirando el test.


### Tablero de compuertas G1 a G7
**Qué se hace:** se muestra el estado de las siete compuertas de avance (protocolo 2.14) tal como lo calcula `scripts/estado_compuertas.py`: el **real** y, por separado, el **simulado** de la demostración.
**Por qué:** las etapas del estudio avanzan por criterios y no por fechas. Una compuerta solo cuenta como cumplida con datos **reales**.
**Paso del protocolo:** 2.14.

In [7]:
tb = leer_json("logs/avance/estado_compuertas.json")
real = pd.DataFrame([{"compuerta": c["id"], "nombre": c["nombre"], "estado": c["estado"], "datos": c["datos"], "criterio": c["criterio"], "nota": c["nota"]} for c in tb["compuertas"] if c["datos"] != "Simulado"])
rotulo(ORIGEN_REAL, f"tablero real: {tb['compuertas_cumplidas_con_datos_reales']} de {tb['total']} compuertas cumplidas con datos reales (generado {tb['generado']})")
display(real)
sim_en_real = [c for c in tb["compuertas"] if c["datos"] == "Simulado"]
if sim_en_real:
    rotulo(ORIGEN_SIMULADO, "compuertas que el tablero real muestra solo con datos simulados (NO cuentan como reales)")
    display(pd.DataFrame([{"compuerta": c["id"], "nombre": c["nombre"], "estado": c["estado"], "datos": c["datos"]} for c in sim_en_real]))
comparar("G1–G7", "compuertas cumplidas con datos reales", tb["compuertas_cumplidas_con_datos_reales"], 5, "Protocolo V1.8 (5 de 7)", tol=0)
estados_real = {c["id"]: c["estado"] for c in tb["compuertas"]}
comparar("G1–G7", "G1–G5 cumplidas, G6 no cumplida", ",".join(f"{k}:{estados_real[k]}" for k in ("G1", "G2", "G3", "G4", "G5", "G6")), "G1:Cumplida,G2:Cumplida,G3:Cumplida,G4:Cumplida,G5:Cumplida,G6:No cumplida", "tablero G6 (PP01–PP07)", tipo="txt")

ruta_sim = BASE / "evidencias/simulado_demostracion/20261005_demostracion/06_tablero/estado_compuertas_SIMULADO.json"
if ruta_sim.exists():
    ts = json.loads(ruta_sim.read_text(encoding="utf-8"))
    sim = pd.DataFrame([{"compuerta": c["id"], "nombre": c["nombre"], "estado": c["estado"], "datos": c.get("datos", "")} for c in ts["compuertas"]])
    rotulo(ORIGEN_SIMULADO, "tablero de la demostración — NINGUNA de estas compuertas cuenta como real")
    display(sim)
else:
    print("(no se encontró el tablero simulado en el paquete)")
print("Los dos tableros NO se suman: real =", tb["compuertas_cumplidas_con_datos_reales"], "de 7; la compuerta G7 solo aparece cumplida en la demostración simulada.")
mostrar_comparaciones(etapas=["G1–G7"], titulo="Esta sección — recalculado frente a reportado")

▌ORIGEN: Real — tablero real: 5 de 7 compuertas cumplidas con datos reales (generado 2026-10-09T08:52:45)


,compuerta,nombre,estado,datos,criterio,nota
0,G1,Lote 1 completo,Cumplida,Real,"«Listo para la Parte B»: ≥ 15 transcritos, 0 sin consentimiento, cada intención ≥ 3 frases",Transcritos 32/15 (≥ 15); intenciones con menos de 3 frases: 0.
1,G2,Parte B ejecutada,Cumplida,Real,Ingesta sin errores bloqueantes y revisión de datos personales marcada por el tesista,Ingesta sin errores bloqueantes y revisión de datos personales marcada.
2,G3,Calidad con lenguaje real,Cumplida,Real,"≤ 2 ciclos de refinamiento y UNA evaluación del test real con F1 macro ≥ 0,75 (V1.6: medida en el lote 2; la del lote 1 queda como «No cumplida, lote 1»)","No cumplida, lote 1. Ciclos de refinamiento: 1/2. Única evaluación del test real: F1 macro = 0.6867 (< 0.75). Medida en lote 2: F1 macro = 0.9072 (≥ 0.75); G3 cumplida con la medición independiente del lote 2 (la del lote 1 queda como «No cumplida, lote 1»)."
3,G4,Respuestas verificadas,Cumplida,Real,"TUPA: Alta pendientes = 0, alertas = 0 y Corregir/Coincide sin confirmar = 0","Alta pendientes: 0; filas con alerta: 0; Corregir o Coincide sin confirmar: 0 (con resultado, propuesto o confirmado: 35 de 44; confirmadas por el tesista: 28)."
4,G5,Modelo congelado,Cumplida,Real,"Modelo congelado válido, después de G3 y G4","Congelado el 2026-10-08 22:17, después de G3 y G4."
5,G6,Pre-piloto,No cumplida,Real,"5–15 sesiones completas y alfa de Cronbach ≥ 0,70",Alfa de Cronbach 0.366 < 0.7: se corrige el instrumento y se repite una sola vez con otro grupo. Con n = 5 el alfa es poco estable (advertencia).


,compuerta,nombre,estado,datos
0,G7,Sesiones,Cumplida (Simulado),Simulado


,compuerta,nombre,estado,datos
0,G1,Lote 1 completo,Cumplida (Simulado),Simulado
1,G2,Parte B ejecutada,En curso (Simulado),Simulado
2,G3,Calidad con lenguaje real,Cumplida (Simulado),Simulado
3,G4,Respuestas verificadas,Pendiente (sin evidencia),—
4,G5,Modelo congelado,En curso (Simulado),Simulado
5,G6,Pre-piloto,Pendiente (sin evidencia),—
6,G7,Sesiones,Cumplida (Simulado),Simulado


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,G1–G7,compuertas cumplidas con datos reales,5,5,Protocolo V1.8 (5 de 7),Sí
1,G1–G7,"G1–G5 cumplidas, G6 no cumplida","G1:Cumplida,G2:Cumplida,G3:Cumplida,G4:Cumplida,G5:Cumplida,G6:No cumplida","G1:Cumplida,G2:Cumplida,G3:Cumplida,G4:Cumplida,G5:Cumplida,G6:No cumplida",tablero G6 (PP01–PP07),Sí


### Cómo leer el resultado de esta sección
- **Real:** G1–G5 «Cumplida» (corpus, ingesta, calidad con lenguaje real en el lote 2, respuestas verificadas y modelo congelado); **G6 «No cumplida»** (alfa 0,366 con n = 5) y **G7 pendiente** (no hay sesiones del piloto). Total: **5 de 7**.
- **Simulado:** G7 aparece «Cumplida (Simulado)» solo porque la demostración usa 60 sesiones inventadas. **No se suma** a las reales.
- G3 es «Cumplida» porque el lote 1 dio F1 0,687 («No cumplida, lote 1») y el lote 2 —prueba independiente, con el diseño declarado antes de medir— dio 0,907. Los dos hechos siguen visibles en la nota de G3.

In [8]:
cierre("P11.1–P11.2b")

P11.1–P11.2b — recalculado frente a reportado: 13 cifras · coinciden: 13 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P11.1,intenciones correctas en el smoke test,46/54,46/54,evidencias/p11_1_pruebas/README.md,Sí
1,P11.1,participantes transcritos del lote 2,25,25,ingesta_lote2_cifras.md,Sí
2,G6,alfa de Cronbach (ítems 1–8),0.365714,0.366,Protocolo V1.8 / tablero G6 / cálculo manual del tesista,Sí
3,G6,alfa: implementación independiente,0.365714,0.365714,misma muestra,Sí
4,G6,sesiones registradas,7,7,tablero G6 (PP01–PP07),Sí
5,G6,sesiones elegibles,5,5,tablero G6,Sí
6,G6,sesiones no elegibles,2,2,tablero G6,Sí
7,G6,incidencias técnicas,0,0,registro del pre-piloto,Sí
8,P15,smoke_compuertas.py,75/75,75/75,ejecución del 2026-10-10 (smoke_asistente suma 3 comprobaciones del modo demo),Sí
9,P15,smoke_piloto.py,51/51,51/51,ejecución del 2026-10-10 (smoke_asistente suma 3 comprobaciones del modo demo),Sí


## (d) Cómo leer el resultado
- **P11.1:** 46/54 intenciones correctas en el *smoke test* de octubre (corpus v2); la CV nativa (0,778) está inflada por fuga y no se usa para G3.
- **Lotes:** el lote 2 trae 25 participantes y 280 frases validadas (Q0001–Q0280); la revisión de etiquetas fue de **una sola revisora** (kappa 1,000 = etiqueta esperada frente a revisión, **no** acuerdo entre revisores).
- **Pre-piloto:** 7 sesiones registradas, 5 elegibles y completas, 2 no elegibles, 0 incidencias técnicas; **alfa 0,366 < 0,70 → G6 «No cumplida»** (con n = 5 es muy inestable).
- **Compuertas reales: 5 de 7.** G7 aparece cumplida solo en la demostración simulada (cuaderno 13).

## Limitaciones
- El alfa de Cronbach con n = 5 no permite concluir qué ítem falla; el protocolo manda corregir el instrumento y repetir una sola vez con otro grupo.
- Las pruebas automáticas usan datos y predictores **falsos**: demuestran que el código detecta lo que debe, no que el modelo funcione.
- No se ejecuta `analizar_piloto.py` sobre el pre-piloto (gastaría la prueba final única del modelo).